# UNET

# 70/10/20


In [ ]:
import os
import glob
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
import cv2
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split

# ----------------------------
# Reproducibility
# ----------------------------
SEED = 45
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ----------------------------
# User settings
# ----------------------------
DATASET_PATH = r"G:\Dataset\Breast Ultrasound Dataset\BUS-UCLM\dataset"
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 8
EPOCHS = 200  # reduced for cross-validation speed
N_FOLDS = 5
MASK_SUFFIX = "_mask"
MASKS_IN_SUBFOLDER = True
AUTOTUNE = tf.data.AUTOTUNE

# ----------------------------
# Find image-mask pairs
# ----------------------------
def find_image_mask_pairs(dataset_path, image_extensions=("png", "jpg", "jpeg"), 
                          mask_suffix=MASK_SUFFIX, masks_in_subfolder=MASKS_IN_SUBFOLDER):
    image_paths, mask_paths = [], []
    class_dirs = [os.path.join(dataset_path, d) for d in os.listdir(dataset_path)
                  if os.path.isdir(os.path.join(dataset_path, d))]
    for cdir in class_dirs:
        for ext in image_extensions:
            for img in glob.glob(os.path.join(cdir, f"*.{ext}")):
                base = os.path.splitext(os.path.basename(img))[0]
                if masks_in_subfolder:
                    mask_candidate = os.path.join(cdir, "masks", base + ".png")
                    if os.path.exists(mask_candidate):
                        image_paths.append(img)
                        mask_paths.append(mask_candidate)
                        continue
                    mask_candidate_jpg = os.path.join(cdir, "masks", base + ".jpg")
                    if os.path.exists(mask_candidate_jpg):
                        image_paths.append(img)
                        mask_paths.append(mask_candidate_jpg)
                        continue
                mask_candidate2 = os.path.join(cdir, base + mask_suffix + ".png")
                mask_candidate2_jpg = os.path.join(cdir, base + mask_suffix + ".jpg")
                if os.path.exists(mask_candidate2):
                    image_paths.append(img); mask_paths.append(mask_candidate2); continue
                if os.path.exists(mask_candidate2_jpg):
                    image_paths.append(img); mask_paths.append(mask_candidate2_jpg); continue
    return image_paths, mask_paths

# ----------------------------
# CLAHE and dataset functions
# ----------------------------
def apply_clahe_numpy(image):
    if image.ndim == 3:
        img_yuv = cv2.cvtColor(image, cv2.COLOR_RGB2YUV)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img_yuv[:, :, 0] = clahe.apply(img_yuv[:, :, 0])
        img_clahe = cv2.cvtColor(img_yuv, cv2.COLOR_YUV2RGB)
        return img_clahe
    return image

def read_image_with_clahe(path, size=IMAGE_SIZE):
    def _read_and_clahe(p):
        path = p.numpy().decode("utf-8")
        img = cv2.imread(path)
        #img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        #img = apply_clahe_numpy(img)
        img = cv2.resize(img, size)
        img = img.astype(np.float32) / 255.0
        return img
    img = tf.py_function(func=_read_and_clahe, inp=[path], Tout=tf.float32)
    img.set_shape([size[0], size[1], 3])
    return img

def read_mask(path, size=IMAGE_SIZE):
    mask = tf.io.read_file(path)
    mask = tf.image.decode_image(mask, channels=1)
    mask.set_shape([None, None, 1])
    mask = tf.image.resize(mask, size, method="nearest")
    mask = tf.cast(mask > 127, tf.float32)
    mean_val = tf.reduce_mean(mask)
    mask = tf.cond(mean_val > 0.5, lambda: 1.0 - mask, lambda: mask)
    return mask

def load_pair(img_path, mask_path):
    img = read_image_with_clahe(img_path)
    mask = read_mask(mask_path)
    return img, mask

def augment(img, mask):
    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.8:
        img = tf.image.random_brightness(img, max_delta=0.1)
        
    # Random rotation ±20°
    angle = tf.random.uniform((), -20, 20) * 3.14159265 / 180
    img = tfa.image.rotate(img, angle, fill_mode='reflect')
    mask = tfa.image.rotate(mask, angle, fill_mode='nearest')
    return img, mask



def make_dataset(image_paths, mask_paths, batch=BATCH_SIZE, training=True):
    ds = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    if training:
        ds = ds.shuffle(buffer_size=len(image_paths))
    ds = ds.map(lambda x, y: load_pair(x, y), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda img, mask: (tf.ensure_shape(img, [IMAGE_SIZE[0], IMAGE_SIZE[1], 3]),
                                   tf.ensure_shape(mask, [IMAGE_SIZE[0], IMAGE_SIZE[1], 1])),
                num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: augment(x, y), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch).prefetch(AUTOTUNE)
    return ds


# ================================================================
# Improved Attention Gate (IAG)
# ================================================================
def IAG(x, g, reduction=4):
    """
    x : skip connection feature map
    g : gating signal from decoder
    reduction : channel reduction factor
    """
    # 1. Reduce channels for attention computation
    channels = x.shape[-1]
    reduced_channels = max(channels // reduction, 1)

    # Linear embeddings
    theta_x = layers.Conv2D(reduced_channels, 1, padding="same")(x)  # [B,H,W,C//r]
    phi_g   = layers.Conv2D(reduced_channels, 1, padding="same")(g)  # [B,H,W,C//r]

    # Resize gating to match skip feature
    if phi_g.shape[1] != theta_x.shape[1]:
        phi_g = tf.image.resize(phi_g, (theta_x.shape[1], theta_x.shape[2]), method='bilinear')

    # -----------------------------
    # 2. Spatial Attention (lightweight)
    # -----------------------------
    # Combine skip + gating
    attn = layers.Add()([theta_x, phi_g])
    attn = layers.Activation("relu")(attn)

    # Apply depthwise conv + pointwise conv for spatial attention
    attn = layers.DepthwiseConv2D(kernel_size=3, padding="same", depth_multiplier=1)(attn)
    attn = layers.Conv2D(1, 1, padding="same", activation="sigmoid")(attn)  # [B,H,W,1]

    # -----------------------------
    # 3. Channel Attention
    # -----------------------------
    # Global average pooling for channel-wise weights
    chn_attn = layers.GlobalAveragePooling2D()(x)
    chn_attn = layers.Dense(reduced_channels, activation="relu")(chn_attn)
    chn_attn = layers.Dense(channels, activation="sigmoid")(chn_attn)
    chn_attn = layers.Reshape((1,1,channels))(chn_attn)

    # -----------------------------
    # 4. Apply combined attention
    # -----------------------------
    out = layers.Multiply()([x, attn])       # spatial
    out = layers.Multiply()([out, chn_attn]) # channel
    out = layers.Add()([out, x])            # residual

    return out


from tensorflow.keras import layers, models
import tensorflow as tf

# ================================================================
# Convolution Block with ECA
# ================================================================
def conv_block(x, filters):
    # Conv 1
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
   
    x = layers.BatchNormalization()(x)

    # Conv 2
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
   
    x = layers.BatchNormalization()(x)

    return x


# ================================================================
# Encoder block
# ================================================================
def encoder_block(x, filters):
    c = conv_block(x, filters)
    p = layers.MaxPooling2D((2, 2))(c)
    return c, p


# ================================================================
# Decoder block with Attention Gate
# ================================================================
def decoder_block(x, skip, filters):
    x = layers.UpSampling2D((2, 2))(x)

    # Apply IAG on skip
    skip_att = IAG(skip, x, filters // 2)

    x = layers.Concatenate()([x, skip_att])
    x = conv_block(x, filters)
    return x


# ================================================================
# Full U-Net with ECA + AG
# ================================================================
def build_unet_with_eca_ag(input_shape=(128,128,3), n_classes=1):
    inputs = layers.Input(shape=input_shape)

    # Encoder
    c1, p1 = encoder_block(inputs, 8)
    c2, p2 = encoder_block(p1, 16)
    c3, p3 = encoder_block(p2, 32)
    c4, p4 = encoder_block(p3, 64)

    # Bottleneck
    b = conv_block(p4, 128)

    # Decoder with AG
    d1 = decoder_block(b, c4, 64)
    d2 = decoder_block(d1, c3, 32)
    d3 = decoder_block(d2, c2, 16)
    d4 = decoder_block(d3, c1, 8)

    # Output layer
    outputs = layers.Conv2D(n_classes, 1, activation='sigmoid')(d4)

    model = models.Model(inputs, outputs, name='UNet_ECA_AG')
    return model



# ---------------------------
# BCE + Jaccard (IoU) Loss
# ---------------------------
def jaccard_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return 1.0 - (intersection + smooth)/(union + smooth)

def bce_jaccard_loss(y_true, y_pred):
    bce = tf.keras.losses.BinaryCrossentropy()(y_true, y_pred)
    jacc = jaccard_loss(y_true, y_pred)
    return bce + jacc

def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2.*intersection + smooth)/(K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.BinaryCrossentropy()(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)


def iou_metric(y_true, y_pred, smooth=1e-6):
    y_pred_pos = tf.cast(y_pred > 0.5, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred_pos)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred_pos) - intersection
    return (intersection + smooth) / (union + smooth)

class MeanIoU(tf.keras.metrics.Metric):
    def __init__(self, name='mean_iou', **kwargs):
        super().__init__(name=name, **kwargs)
        self.total = self.add_weight(name='total', initializer='zeros')
        self.count = self.add_weight(name='count', initializer='zeros')
    def update_state(self, y_true, y_pred, sample_weight=None):
        iou = iou_metric(y_true, y_pred)
        self.total.assign_add(iou)
        self.count.assign_add(1.0)
    def result(self):
        return self.total / (self.count + 1e-12)
    def reset_states(self):
        self.total.assign(0.0)
        self.count.assign(0.0)

import tensorflow as tf

def tversky_index(y_true, y_pred, alpha=0.7, smooth=1e-6):
    """
    Compute Tversky index.
    alpha: weight for false negatives (FN)
    1-alpha: weight for false positives (FP)
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    y_true_flat = tf.reshape(y_true, [-1])
    y_pred_flat = tf.reshape(y_pred, [-1])
    
    TP = tf.reduce_sum(y_true_flat * y_pred_flat)
    FP = tf.reduce_sum((1 - y_true_flat) * y_pred_flat)
    FN = tf.reduce_sum(y_true_flat * (1 - y_pred_flat))
    
    return (TP + smooth) / (TP + alpha * FN + (1 - alpha) * FP + smooth)

def focal_tversky_loss(y_true, y_pred, alpha=0.7, gamma=0.75):
    """
    Focal Tversky Loss
    gamma > 0 : focusing parameter to penalize hard-to-segment pixels
    """
    T_index = tversky_index(y_true, y_pred, alpha=alpha)
    return tf.pow((1 - T_index), gamma)

# Optional: combined loss with BCE
def bce_focal_tversky_loss(y_true, y_pred, alpha=0.7, gamma=0.75):
    bce = tf.keras.losses.BinaryCrossentropy()(y_true, y_pred)
    ft_loss = focal_tversky_loss(y_true, y_pred, alpha=alpha, gamma=gamma)
    return bce + ft_loss


import tensorflow as tf

def tversky_index_main(y_true, y_pred, alpha=0.7, beta=0.3, smooth=1e-6):
    """
    Compute Tversky index per sample.
    
    y_true, y_pred: tensors of shape [batch, H, W, C]
    alpha: weight for false negatives (FN)
    beta: weight for false positives (FP)
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # Flatten per sample
    y_true_flat = tf.reshape(y_true, [tf.shape(y_true)[0], -1])
    y_pred_flat = tf.reshape(y_pred, [tf.shape(y_pred)[0], -1])

    TP = tf.reduce_sum(y_true_flat * y_pred_flat, axis=1)
    FP = tf.reduce_sum((1 - y_true_flat) * y_pred_flat, axis=1)
    FN = tf.reduce_sum(y_true_flat * (1 - y_pred_flat), axis=1)

    T_index = (TP + smooth) / (TP + alpha * FN + beta * FP + smooth)
    return T_index

def focal_tversky_loss_main(y_true, y_pred, alpha=0.7, beta=0.3, gamma=4/3, smooth=1e-6):
    """
    Focal Tversky Loss
    gamma > 1 focuses training on hard pixels
    """
    T_index = tversky_index_main(y_true, y_pred, alpha=alpha, beta=beta, smooth=smooth)
    FTL = tf.pow((1 - T_index), gamma)
    return tf.reduce_mean(FTL)  # Average over batch

def dice_focal_tversky_loss(y_true, y_pred,
                            alpha=0.7, beta=0.3, gamma=0.75,
                            w_dice=0.5, w_ft=0.5):
    dl = dice_loss(y_true, y_pred)
    ftl = focal_tversky_loss(y_true, y_pred, alpha,  gamma)
    return w_dice * dl + w_ft * ftl
import numpy as np
import cv2
import os

def save_segmentation_triplet(original, gt_mask, pred_mask, save_path):
    """
    original: (H,W,3) RGB image in [0,1]
    gt_mask: (H,W,1) binary GT mask 0/1
    pred_mask: (H,W,1) predicted probabilities 0–1 (will threshold at 0.5)
    save_path: full output path (.png)
    """

    # Convert original image to uint8
    orig = (original * 255).astype(np.uint8)

    # Keep gt as 0/1 for logic
    gt_logic = gt_mask.squeeze().astype(np.uint8)
    pred_bin = (pred_mask.squeeze() > 0.5).astype(np.uint8)

    # ---------- COLOR ENCODING ----------
    # True Positive = White
    # False Positive = Red
    # False Negative = Green
    
    H, W = gt_logic.shape
    colored = np.zeros((H, W, 3), np.uint8)

    # True Positive
    TP = (gt_logic == 1) & (pred_bin == 1)
    colored[TP] = [255, 255, 255]  # white in BGR

    # False Positive
    FP = (gt_logic == 0) & (pred_bin == 1)
    colored[FP] = [0, 0, 255]  # red in BGR

    # False Negative
    FN = (gt_logic == 1) & (pred_bin == 0)
    colored[FN] = [0, 255, 0]  # green in BGR

    # Ground truth mask for display (0–255)
    gt_rgb = cv2.cvtColor((gt_logic * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)

    # ---------- CONCATENATE ----------
    triplet = np.concatenate([orig, gt_rgb, colored], axis=1)

    # ---------- ADD WHITE VERTICAL LINES ----------
    line_thickness = 2  # thickness of the vertical line
    total_width = triplet.shape[1]
    single_width = total_width // 3

    # Draw first line (after original)
    triplet[:, single_width:single_width+line_thickness] = 255  # white line

    # Draw second line (after gt mask)
    triplet[:, 2*single_width:2*single_width+line_thickness] = 255  # white line

    # Save
    cv2.imwrite(save_path, triplet)



# ----------------------------
# Main: Fixed 70/10/20 Split
# ----------------------------
if __name__ == "__main__":
    imgs, masks = find_image_mask_pairs(DATASET_PATH)
    print(f"Found {len(imgs)} image-mask pairs.")
    imgs = np.array(imgs)
    masks = np.array(masks)

    # Split into 70% train, 30% temp
    train_imgs, temp_imgs, train_masks, temp_masks = train_test_split(
        imgs, masks, test_size=0.3, random_state=SEED
    )

    # Split temp into 10% val, 20% test (relative to total)
    val_ratio = 10 / 30  # 10% of total = 1/3 of temp
    val_imgs, test_imgs, val_masks, test_masks = train_test_split(
        temp_imgs, temp_masks, test_size=(20/30), random_state=SEED
    )

    print(f"Train: {len(train_imgs)}, Val: {len(val_imgs)}, Test: {len(test_imgs)}")

    # Create datasets
    train_ds = make_dataset(train_imgs, train_masks, batch=BATCH_SIZE, training=True)
    val_ds = make_dataset(val_imgs, val_masks, batch=BATCH_SIZE, training=False)
    test_ds = make_dataset(test_imgs, test_masks, batch=BATCH_SIZE, training=False)
    
    

    # Build and compile model
    model = build_simple_unet()
    model.summary()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss= focal_tversky_loss,
        metrics=[
            dice_coef,
            tf.keras.metrics.MeanIoU(num_classes=2),
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall")
        ]
    )

    # Callbacks
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_dice_coef", patience=15, mode="max", restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_dice_coef", factor=0.1, patience=10, verbose=1, mode="max", min_lr=1e-7),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=f"G:/Dataset/Breast Ultrasound Dataset/Model/best.h5",
            monitor="val_dice_coef",
            save_best_only=True,
            mode="max",
            verbose=1
        )
    ]

    # Train model
    #model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=1)
    
    
    # Load best saved weights
    BEST_WEIGHTS = "G:/Dataset/Breast Ultrasound Dataset/Model/best.h5"
    model.load_weights(BEST_WEIGHTS)
    print("Loaded best weights successfully!")
    
    # Evaluate on validation
    val_results = model.evaluate(val_ds, verbose=0)
    print("Validation metrics:", dict(zip(model.metrics_names, val_results)))

    # Evaluate on test
    test_results = model.evaluate(test_ds, verbose=0)
    print("Test metrics:", dict(zip(model.metrics_names, test_results)))
    
    
    SAVE_DIR = f"G:/Dataset/Breast Ultrasound Dataset/results/best/"
    os.makedirs(SAVE_DIR, exist_ok=True)

    idx = 0

    for batch_imgs, batch_masks in test_ds:

        for img, gt in zip(batch_imgs, batch_masks):

            # Convert tensors → numpy
            img = img.numpy()            # (H,W,3) normalized
            gt = gt.numpy()              # (H,W,1) 0/1

            # Predict
            pred = model.predict(img[np.newaxis, ...], verbose=0)[0]
            pred = np.clip(pred, 0, 1)

            out_path = os.path.join(SAVE_DIR, f"{idx:03d}.png")
            save_segmentation_triplet(img, gt, pred, out_path)

            idx += 1

  
